In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict

In [2]:
ROOT = Path.cwd()
QUESTION_PATH = ROOT / "all_questions.jsonl"
ROOT

WindowsPath('e:/2026/字节和我的心脏只有一个可以跳动/rag_core')

In [84]:
def load_jsonl(path) -> list[dict]:
    rows  = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows
    
questions = load_jsonl(QUESTION_PATH)
print(type(questions))
print(questions[0])
print(type(questions[0]))
print(len(questions))
print(questions[0].keys())

<class 'list'>
{'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'role': 'cpp', 'role_label': 'C++工程师', 'topic': 'C++ Syntax', 'topic_id': 'cpp:cxx_syntax', 'question_type': 'project_deep_dive', 'difficulty': 'hard', 'question': '请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。', 'expected_answer': '项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。', 'reference_points': ['项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。', '方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。', '权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。', '优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。'], 'follow_up_angles': ['如果项目规模扩大，如何管理模板代码的可读性和维护性？', '在现代 C++（C++20）中，你会如何用概念（concepts）重构此模板代码？', '请分享一个因模板误用导致的隐蔽运行时错误案例及调试过程。'], 'common_mistakes': ['只描述项目背景，没有说清自己的职责和贡献。', '没有量化效果，也没有说明方案取舍。', '遇到问题时只给结果，不讲排查路径和

In [185]:
from langchain_core.documents import Document
def json_to_question_document(obj: dict, source_file: str) -> Document:
    """
    把 _questions.jsonl 的一行转换成题目 Document。
    用于 interview_questions collection。
    """
    question_id = obj.get("question_id", "")
    role = obj.get("role", "")
    topic = obj.get("topic", "")
    topic_id = obj.get("topic_id", "")
    question_type = obj.get("question_type", "")
    difficulty = obj.get("difficulty", "")
    question = obj.get("question", "")
    expected_answer = obj.get("expected_answer", "")
    reference_points = obj.get("reference_points", [])
    follow_up_angles = obj.get("follow_up_angles", [])
    common_mistakes = obj.get("common_mistakes", [])
    tags = obj.get("tags", [])
    assesses_topic_ids = obj.get("assesses_topic_ids", [])

    page_content = (
        f"[文档类型]: interview_question\n"
        f"[题目ID]: {question_id}\n"
        f"[岗位]: {role}\n"
        f"[知识点]: {topic}\n"
        f"[主题ID]: {topic_id}\n"
        f"[题型]: {question_type}\n"
        f"[难度]: {difficulty}\n"
        f"[题目]: {question}\n"
        f"[参考答案]: {expected_answer}\n"
        # f"[评分要点]: {_join_list(reference_points)}\n"
        # f"[追问方向]: {_join_list(follow_up_angles)}\n"
        # f"[常见错误]: {_join_list(common_mistakes)}\n"
        # f"[考察主题]: {_join_list(assesses_topic_ids)}\n"
        # f"[标签]: {_join_list(tags)}\n"
    )

    metadata = {
        "doc_type": "interview_question",
        "source": source_file,
        "question_id": question_id,
        "question": question, 
        "role": role,
        "topic": topic,
        "topic_id": topic_id,
        "question_type": question_type,
        "difficulty": difficulty,
        "source_file": obj.get("source_file", source_file),
        "source_line": int(obj.get("source_line", 0) or 0),
        "expect_answer": expected_answer,
        "follow_up_angles": json.dumps(follow_up_angles, ensure_ascii=False),
    }

    return Document(page_content=page_content, metadata=metadata)

In [186]:
def load_documents_and_ids_from_jsonl(file_path:Path) -> list[dict]:
    """
    打开文件地址，读取文件内容，并把每一行的 JSON 字符串转换成 Document 对象，最后返回一个 Document 对象的列表。
    """
    documents = []
    ids = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            doc = json_to_question_document(obj,source_file=file_path.name)
            documents.append(doc)
            doc_id = f"{file_path.stem}_{len(documents)}"
            ids.append(doc_id)
            
    return documents,ids

In [187]:
docs, ids = load_documents_and_ids_from_jsonl(QUESTION_PATH)
print(docs[0])
print(type(docs))
print(ids[0])

page_content='[文档类型]: interview_question
[题目ID]: cpp_c++_syntax_project_deep_dive_1c18e81950
[岗位]: cpp
[知识点]: C++ Syntax
[主题ID]: cpp:cxx_syntax
[题型]: project_deep_dive
[难度]: hard
[题目]: 请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。
[参考答案]: 项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。
' metadata={'doc_type': 'interview_question', 'source': 'all_questions.jsonl', 'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'question': '请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。', 'role': 'cpp', 'topic': 'C++ Syntax', 'topic_id': 'cpp:cxx_syntax', 'question_type': 'project_deep_dive', 'difficulty': 'hard', 'source_file': 'cpp_project.jsonl', 'source_line': 1, 'expect_answer': '项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
def init_chroma_db(persist_dir, collection_name):
    emb = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs = {"device":"cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )

    db = Chroma(
        persist_directory=persist_dir,
        collection_name=collection_name,
        embedding_function=emb,
    )  
    return db

def add_documents_to_chroma(db, documents, ids):
    db.add_documents(documents, ids=ids)
    return db

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [189]:
persist_dir = "chroma_all_v2"
collection_name = "question"
db = init_chroma_db(
    persist_dir=persist_dir,
    collection_name=collection_name,
)
# db = add_documents_to_chroma(
#     db=db,
#     documents=docs,
#     ids=ids,
# )
batch_size = 1000
for i in range(0,len(docs),batch_size):
    batch_docs = docs[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]

    db.add_documents(batch_docs, ids=batch_ids)
print(db._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2553.65it/s]


8350


In [6]:
persist_dir = "chroma_all_v2"
collection_name = "question"
db = init_chroma_db(
    persist_dir=persist_dir,
    collection_name=collection_name,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2995.52it/s]


In [7]:
print(db._collection.count())

8350


In [91]:
result = db._collection.get(
    ids = ["all_questions_1"]
)
print(type(result))
for doc_id , doc_text , mete in zip(result["ids"] , result["documents"],result["metadatas"]):
    print(f"ID: {doc_id}\npage_content:\n{doc_text}\nmetedata: {mete}\n")

<class 'dict'>
ID: all_questions_1
page_content:
[文档类型]: interview_question
[题目ID]: cpp_c++_syntax_project_deep_dive_1c18e81950
[岗位]: cpp
[知识点]: C++ Syntax
[主题ID]: cpp:cxx_syntax
[题型]: project_deep_dive
[难度]: hard
[题目]: 请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。
[参考答案]: 项目背景：简述问题场景，如需要在编译期进行类型计算或策略选择。；方案选择：对比模板元编程与运行时多态（如虚函数）的优缺点，强调编译期计算的优势（如零运行时开销）。；权衡与排障：讨论编译错误（如模板递归深度限制）或性能瓶颈（如代码膨胀），并说明解决方法。；优化与复盘：如何通过 SFINAE、C++17 的 if constexpr 或概念（concepts）简化模板代码，并评估实际性能提升。

metedata: {'topic': 'C++ Syntax', 'source_file': 'cpp_project.jsonl', 'difficulty': 'hard', 'question': '请描述一个你在实际项目中使用 C++ 模板元编程（TMP）解决复杂问题的经历。具体说明你为何选择模板元编程而非运行时多态，并阐述方案权衡、遇到的编译错误或性能问题，以及最终如何优化。', 'topic_id': 'cpp:cxx_syntax', 'source': 'all_questions.jsonl', 'doc_type': 'interview_question', 'source_line': 1, 'question_id': 'cpp_c++_syntax_project_deep_dive_1c18e81950', 'role': 'cpp', 'question_type': 'project_deep_dive'}



In [31]:
class InterviewPromptSet:
    evaluate_prompt: str = """ 
    你是一个严格但公平的技术面试官。请基于当前题目、候选人主回答、追问回答、参考答案、评分要点、常见错误和 RAG 检索上下文，给出单题综合评分。

    评分要求：
    1. 分数必须是 0-100 的整数。
    2. 重点评价：技术正确性、关键点覆盖度、工程经验深度、表达清晰度。
    3. 如果候选人回答存在概念错误、编造、答非所问，必须写入 mistakes。
    4. 不要机械照抄参考答案评分；如果候选人回答技术上合理，可以认可。
    5. 只输出 JSON，不要输出 Markdown，不要输出额外解释。

    输出 JSON 格式如下：
    {{
    "score": 75,
    "reason": "一句到三句话说明评分理由",
    "hit_points": ["候选人命中的要点"],
    "missing_points": ["候选人缺失的要点"],
    "mistakes": ["候选人的明显错误，没有则为空列表"],
    "suggestion": "给候选人的改进建议"
    }}
    【当前题目】
    {main_question}

    【候选人主回答】
    {main_answer}

    【追问】
    {followup_question}

    【候选人追问回答】
    {followup_answer}

    【参考答案】
    {expected_answer}
    """,
    followup_prompt: str = """
    你是一个严格但公平的技术面试官。请基于当前题目、候选人的主回答、参考答案和追问方向，生成一个追问问题。

    要求：
    1. 只输出一个问题。
    2. 不要给答案。
    3. 不要解释为什么这样问。
    4. 追问必须围绕当前题、用户回答中的薄弱点，或给定追问方向。
    5. 如果用户回答过于笼统，优先追问具体工程细节、边界条件、异常处理、性能瓶颈或方案取舍。
    6. 不要问多个问题。

    【当前题目】
    {main_question}

    【候选人主回答】
    {main_answer}

    【参考答案】
    {expected_answer}

    【可选追问方向】
    {follow_up_angles}

    【RAG 检索上下文】
    {retrieved_context}
    """
InterviewPromptSet.followup_prompt

'\n    你是一个严格但公平的技术面试官。请基于当前题目、候选人的主回答、参考答案和追问方向，生成一个追问问题。\n\n    要求：\n    1. 只输出一个问题。\n    2. 不要给答案。\n    3. 不要解释为什么这样问。\n    4. 追问必须围绕当前题、用户回答中的薄弱点，或给定追问方向。\n    5. 如果用户回答过于笼统，优先追问具体工程细节、边界条件、异常处理、性能瓶颈或方案取舍。\n    6. 不要问多个问题。\n\n    【当前题目】\n    {main_question}\n\n    【候选人主回答】\n    {main_answer}\n\n    【参考答案】\n    {expected_answer}\n\n    【可选追问方向】\n    {follow_up_angles}\n\n    【RAG 检索上下文】\n    {retrieved_context}\n    '

In [8]:
from dataclasses import dataclass , field
# dataclass注意 没有默认值的字段，不能放在有默认值字段的后面。
# non-default argument 'question' follows default argument
@dataclass
class Question:
    question_id: str
    topic: str
    difficulty: str 
    question: str
    expected_answer: str = ""
    follow_up_angles: list[str] = field(default_factory=list)

In [13]:
from typing import Any
@dataclass
class valMaterials:
    expected_answer: str = ""
    reference_points: list[str] = field(default_factory=list)
    common_mistakes: list[str] = field(default_factory=list)
    rubric: dict[str, int] = field(default_factory=dict)
    retrieved_chunks: list[Any] = field(default_factory=list)
@dataclass   
class FollowupMaterials:
    expected_answer: str = ""
    follow_up_angles: list[str] = field(default_factory=list)
    retrieved_context: str = "无"
""" 
单题评分结构体
主问题和追问都可以使用
"""
@dataclass
class valResult:
    score: int
    reason: str
    hit_points: list[str] = field(default_factory=list)
    missing_points: list[str] = field(default_factory=list)
    mistakes: list[str] = field(default_factory=list)
    suggestion: str | None = None
""" 
list是可变对象 如果多个实例共享一个默认列表 一个对象修改列表时 其他对象可能被一起影响到
class EvaluationResult:
    hit_points = []
a = EvaluationResult()
b = EvaluationResult()
当我修改a.hit_points.append("答对了 KV Cache")
b也会被修改

所以dataclass会阻止这样定义 
hit_points: list[str] = field(default_factory=list)
这样都会单独调用一次 list() 生成一个全新的空列表
"""

' \nlist是可变对象 如果多个实例共享一个默认列表 一个对象修改列表时 其他对象可能被一起影响到\nclass EvaluationResult:\n    hit_points = []\na = EvaluationResult()\nb = EvaluationResult()\n当我修改a.hit_points.append("答对了 KV Cache")\nb也会被修改\n\n所以dataclass会阻止这样定义 \nhit_points: list[str] = field(default_factory=list)\n这样都会单独调用一次 list() 生成一个全新的空列表\n'

In [14]:
""" 
完整的题单的数据结构 plan - plan中的一道题目 - 题目
"""
@dataclass
class InterviewPlanItem:
    index: int
    question_id: str
    topic: str
    question: Question
    
@dataclass
class InterviewPlan:
    plan_id: str
    topics: list[str]
    num_questions: int
    items: list[InterviewPlanItem]

In [15]:
""" 
一道题的完整记录
"""

""" 
@dataclass 会自动生成一个init方法 大致等价于 
    self.session_id = session_id 
    ....
创建的时候就可以 session = InterviewSession(
    session_id="xxx",
    user_id="u001",
    status="in_progress",
    current_index=0,
)
注意 不可以 session = InterviewSession() 这被称为空参构造 没有默认值的字段都是必填参数
"""
@dataclass
class InterviewTurn:
    index: int
    question_id: str
    topic: str

    main_question: str
    main_answer: str | None = None
    followup_question: str | None = None
    followup_answer: str | None = None

    score: int | None = None
    evaluation: valResult | None = None
    status: str = "not_started" # waiting_main_answer / waiting_followup_answer / completed

In [16]:
""" 
用来记录完整面试流程
plan 记录了完整的面试题单 
turns 记录了每道题目的完整记录/问答情况 包括评分结果
session.plan.items[session.current_index] 就是当前题目
"""
@dataclass
class InterviewSession:
    session_id: str
    user_id: str
    plan : InterviewPlan

    current_index: int
    created_time: str
    updated_time: str
    finished_time: str | None = None
    final_report: dict = field(default_factory=dict)
    turns: list[InterviewTurn] = field(default_factory=list)
    
    status: str = "not_started" # in_progress / completed
# @dataclass
# class InterviewReport:

In [17]:
import uuid
from datetime import datetime
def init_interview_session(user_id:str, interview_plan:InterviewPlan) -> InterviewSession:
    
    turns = []
    for item in interview_plan.items:
        turn = InterviewTurn(
            index=item.index,
            question_id=item.question_id,
            topic=item.topic,
            main_question=item.question.question,
            status="not_started"
        )
        turns.append(turn)
    if turns:
        turns[0].status = "waiting_main_answer"
    # uuid是python标准库 用来生成通用唯一标识符
    # pythonuuid.uuid4() 会随机生成一个唯一id
    return InterviewSession(
        session_id=str(uuid.uuid4()),
        user_id=user_id,
        plan=interview_plan,
        current_index=0,
        turns=turns,
        created_time=datetime.now().isoformat(),
        updated_time=datetime.now().isoformat(),
        status="in_progress",
    )


In [18]:
def build_materials_from_turn(session: InterviewSession, turn: InterviewTurn) -> valMaterials:
    question = session.plan.items[turn.index].question

    return valMaterials(
        expected_answer=question.expected_answer,
    )

def build_followup_materials(session: InterviewSession, turn: InterviewTurn) -> FollowupMaterials:
    question = session.plan.items[turn.index].question

    return FollowupMaterials(
        expected_answer=question.expected_answer,
        follow_up_angles=question.follow_up_angles,
        retrieved_context="无",
    )

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from datetime import datetime
class InterviewEngine:
    def _now() -> str:
        return datetime.now().isoformat(timespec="seconds")
    

    def get_current_turn(session:InterviewSession) -> InterviewTurn:
        """ 
        获取当前正在进行的那一道题
        检查 session.status 是不是 in_progress
        检查 current_index 有没有越界
        返回 session.turns[session.current_index]
        """
        if session.status != "in_progress":
            raise ValueError(f"当前面试状态不是 in_progress 而是 {session.status}")
        if session.current_index >= len(session.turns):
            raise IndexError( f"current_index 越界: {session.current_index}")
        return session.turns[session.current_index]
    
    def get_current_question(session: InterviewSession) -> str:
        """ 
        返回当前应该展示给用户的问题
        turn.status == waiting_main_answer 返回 turn.question
        turn.status == waiting_followup_answer 返回 turn.followup_question
        """
        if session.status == "completed":
            return None
        turn = InterviewEngine.get_current_turn(session)
        if turn.status == "waiting_main_answer":
            return turn.main_question
        if turn.status == "waiting_followup_answer":
            return turn.followup_question
        raise ValueError(f"当前 turn 状态不允许获取问题: {turn.status}")
    
    def submit_main_answer(session:InterviewSession, answer: str, llm:ChatOpenAI) -> InterviewSession:
        """  
        提交主问题回答，然后生成追问
        1. 检查 session.status == in_progress
        2. 获取 current_turn
        3. 检查 turn.status == waiting_main_answer
        4. 保存 turn.answer = answer
        5. 调用 fake_generate_followup
        6. 保存 turn.followup_question
        7. 修改 turn.status = waiting_followup_answer
        8. 更新 session.updated_time
        9. 返回 session
        """
        if not answer or not answer.strip():
            raise ValueError("追问回答不能为空")
        turn = InterviewEngine.get_current_turn(session)
        if turn.status != "waiting_main_answer":
            raise ValueError(f"当前 turn 状态不允许提交主回答: {turn.status}")
        turn.main_answer = answer
        # turn.followup_question = InterviewEngine.generate_followup(turn)
        materials = build_followup_materials(session, turn)
        turn.followup_question = generate_followup_question(turn, materials, llm=llm)
        turn.status = "waiting_followup_answer"
        session.updated_time = InterviewEngine._now()
        return session

    def submit_followup_answer(session:InterviewSession, answer: str, llm: ChatOpenAI | None = None) -> InterviewSession:
        """ 
        1. 检查 session.status == in_progress
        2. 获取 current_turn
        3. 检查 turn.status == waiting_followup_answer
        4. 保存 turn.followup_answer = answer
        5. 调用 fake_evaluate_turn，综合主回答和追问回答评分
        6. 保存 turn.evaluation
        7. 修改 turn.status = completed
        8. 如果还有下一题：
            session.current_index += 1
            下一题 status = waiting_main_answer
        如果没有下一题：
            session.status = completed
            session.finished_time = 当前时间
            session.final_report = fake_generate_report(session)
        9. 更新 session.updated_time
        10. 返回 session
        """
        if not answer or not answer.strip():
            raise ValueError("追问回答不能为空")
        turn = InterviewEngine.get_current_turn(session)
        if turn.status != "waiting_followup_answer":
            raise ValueError(f"当前 turn 状态不允许提交追问回答: {turn.status}")
        turn.followup_answer = answer
        # turn.evaluation = InterviewEngine.evaluate_turn(turn)
        materials = build_materials_from_turn(session, turn)
        turn.evaluation = evaluate_turn(turn, materials, llm=llm)
        turn.score = turn.evaluation.score
        turn.status = "completed"
        
        have_next = session.current_index + 1 < len(session.turns)
        if have_next:
            session.current_index += 1 
            next_turn = session.turns[session.current_index]
            next_turn.status = "waiting_main_answer"
        else:
            session.status = "completed"
            session.finished_time = InterviewEngine._now()
            session.final_report = InterviewEngine.generate_report(session)  

        session.updated_time = InterviewEngine._now()
        return session
    
    def generate_followup(turn: InterviewTurn) -> str:
        """ 
        根据当前问题和主回答生成一个假的追问
        你刚才回答了「xxx」，请进一步说明其中最关键的一个点。
        """
        question_main = turn.main_question
        answer_main =  turn.main_answer
        question_follow = turn.followup_question
        answer_follow = turn.followup_answer
        query = f"原问题:{question_main}\n 原问题回答:{answer_main}"

        return query
    
    # def evaluate_turn(turn: InterviewTurn) -> EvaluationResult:
    #     """ 
    #     综合主回答和追问回答，生成一个最终评分
    #     turn.question
    #     turn.answer
    #     turn.followup_question
    #     turn.followup_answer
    #     拼接为query 准备好llm所需的内容
    #     """
    #     return EvaluationResult(score=0, reason="TODO: 评分逻辑待实现")
    
    def generate_report(session: InterviewSession) -> dict:
        """ 
        面试结束后，根据所有 turn 的 evaluation 生成报告
        """
        return {
            "session_id": session.session_id,
            "status": session.status,
            "message": "TODO: 最终报告待实现",
        }

In [21]:
def parse_json_list(value) -> list[str]:
    if value is None:
        return []

    if isinstance(value, list):
        return [str(x) for x in value]

    if isinstance(value, str):
        if not value.strip():
            return []

        try:
            data = json.loads(value)
        except json.JSONDecodeError:
            return [value]

        if isinstance(data, list):
            return [str(x) for x in data]

        return [str(data)]

    return [str(value)]

In [22]:
def build_interview_plan(db, topics: list[str], num_questions, difficulty: str="medium") -> InterviewPlan:
    random_seed = 1
    rng = random.Random(random_seed)
    all_candidate_ids: list[str] = []
    topic_to_ids: dict[str, list[str]] = {}
    for topic in topics:
        where = {
            "$and":[
                {"topic":topic},
                {"difficulty":difficulty}
            ]
        }
        result = db.get(where = where,
                        include = ["metadatas"])
        ids = result.get("ids",[])
        all_candidate_ids.extend(ids)
        topic_to_ids[topic] = ids
    all_candidate_ids = list(dict.fromkeys(all_candidate_ids))
    random.shuffle(all_candidate_ids)
    selected_ids: list[str] = []
    for topic in topics:
        ids = topic_to_ids.get(topic, [])
        for doc_id in ids:
            if len(selected_ids) >= num_questions:
                break
            if doc_id not in selected_ids:
                selected_ids.append(doc_id)
                break
    # 如果未满数量
    for doc_id in all_candidate_ids:
        if len(selected_ids) >= num_questions:
            break
        if doc_id not in selected_ids:
            selected_ids.append(doc_id)
    if len(selected_ids) < num_questions:
        print("剩余题目数量不足")

    selected_result = db.get(
        ids = selected_ids,
        include = ["documents", "metadatas"]
    )
    ids = selected_result.get("ids",[])
    documents = selected_result.get("documents", [])
    metadatas = selected_result.get("metadatas", [])

    # 构造 InterviewPlanItem
    items = []
    for index, (doc_id, document, metadata) in enumerate(
        zip(ids, documents, metadatas)
    ):
        # 注意哦 这里的metedata是可以get的 它是 dict类型
        # 但是document不是 它是完全的文字 组织过了的 无法进行数据获取
        metadata = metadata or {}
        question_id = metadata.get("question_id",doc_id)
        question_text = metadata.get("question", None)
        topic = metadata.get("topic", "")
        difficult = metadata.get("difficulty", "")
        expected_answer = metadata.get("expect_answer", "")
        follow_up_angles = parse_json_list(metadata.get("follow_up_angles", "[]"))
        question = Question(
            question_id = question_id,
            topic = topic,
            difficulty = difficult,
            question = question_text,
            expected_answer=expected_answer,
            follow_up_angles=follow_up_angles,
        )
        item = InterviewPlanItem(
            index = index, # 1-5 题目序号索引
            question_id = question_id,
            topic = topic,
            question = question
        )
        items.append(item)
    return InterviewPlan(
        plan_id = str(uuid.uuid4()),
        topics = topics,
        num_questions = num_questions,
        items = items
    )

In [23]:
import uuid
topics = ["C++11","C++14"]
difficulty = "medium"
plan = build_interview_plan(db, topics, 5, difficulty=difficulty)
print(plan)
print(type(plan))
print(plan.plan_id)
print(plan.topics)
print(plan.num_questions)
print(plan.items)
print(plan.items[0].index)
print(plan.items[0].question.expected_answer)
print(plan.items[1].question_id)
print(plan.items[2].question.question)
# 现在没有题目 因为metadata中没有保存question 
# 现在有了id集合 后续考虑加一个数据库连接 mongodb 存储id - question

InterviewPlan(plan_id='f3e24206-d05f-44a2-8786-1a875646ad02', topics=['C++11', 'C++14'], num_questions=5, items=[InterviewPlanItem(index=0, question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', question=Question(question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', difficulty='medium', question='请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。', expected_answer='选择`std::async`而非手动创建线程的原因，如资源管理与异常安全；使用`std::future`获取结果时遇到的阻塞问题及如何通过`std::shared_future`优化；性能瓶颈分析，如线程池资源不足或任务粒度不当；优化措施，如调整任务队列或结合C++11的移动语义减少拷贝；复盘：权衡异步与同步方案的适用场景', follow_up_angles=['如何处理`std::future`的异常传播？', '在高并发场景下，如何设计任务调度器以避免线程饥饿？', 'C++11的异步机制与后续版本（如C++17的并行算法）有何改进？'])), InterviewPlanItem(index=1, question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', question=Question(question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', difficulty='medium', question='请描述一个你使用 C++14 特性（如泛型 lambda、变量模板或 constexpr 改进）优化过的真实项目模块。重点说明你为何选择该特性、如何权衡设计

In [24]:
session = init_interview_session(user_id= "aaa1", interview_plan = plan)

In [25]:
question = InterviewEngine.get_current_question(session)
print(question)

请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。


In [26]:
print(session.session_id)
print(session.plan)
print(session.plan.items[0])
print(session.updated_time)
print(session.status)

turn = session.turns[0]
print(turn.main_question)
print(turn.main_answer)
print(turn.status)

c6a0e59b-c317-45f6-9364-cd6c77782cba
InterviewPlan(plan_id='f3e24206-d05f-44a2-8786-1a875646ad02', topics=['C++11', 'C++14'], num_questions=5, items=[InterviewPlanItem(index=0, question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', question=Question(question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', difficulty='medium', question='请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。', expected_answer='选择`std::async`而非手动创建线程的原因，如资源管理与异常安全；使用`std::future`获取结果时遇到的阻塞问题及如何通过`std::shared_future`优化；性能瓶颈分析，如线程池资源不足或任务粒度不当；优化措施，如调整任务队列或结合C++11的移动语义减少拷贝；复盘：权衡异步与同步方案的适用场景', follow_up_angles=['如何处理`std::future`的异常传播？', '在高并发场景下，如何设计任务调度器以避免线程饥饿？', 'C++11的异步机制与后续版本（如C++17的并行算法）有何改进？'])), InterviewPlanItem(index=1, question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', question=Question(question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', difficulty='medium', question='请描述一个你使用 C++14 特性（如泛型 lambda、变量模板或 conste

In [28]:
import os
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import Any

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
def build_llm() -> ChatOpenAI:
    load_dotenv()
    return ChatOpenAI(
        model="deepseek-v4-flash",
        api_key=os.environ["DEEPSEEK_API_KEY"], # 从环境变量里读取你的 API key 使用之前需要load_dotenv()
        base_url="https://api.deepseek.com", # DeepSeek 的 OpenAI 兼容接口地址
        temperature=0,
    )

In [29]:
llm = build_llm()

In [32]:
def _as_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]
def evaluate_turn(turn: InterviewTurn, materials: valMaterials, llm:ChatOpenAI | None=None) -> valResult:
    if not turn.main_answer or not turn.main_answer.strip():
        raise ValueError("主回答不能为空，不能进行评分")
    if not turn.followup_answer or not turn.followup_answer.strip():
        raise ValueError("追问回答不能为空，不能进行评分")
    
    materials = materials or valMaterials()
    llm = llm or build_llm()
    prompt = ChatPromptTemplate.from_template(InterviewPromptSet.evaluate_prompt)
    parser = JsonOutputParser()
    chain = prompt | llm | parser

    data = chain.invoke(
        {
            "main_question": turn.main_question,
            "main_answer": turn.main_answer,
            "followup_question": turn.followup_question or "无",
            "followup_answer": turn.followup_answer,
            "expected_answer": materials.expected_answer or "无",
        }
    )
    score = int(data.get("score", 0))
    return valResult(
        score=score,
        reason=str(data.get("reason", "")),
        hit_points=_as_list(data.get("hit_points")),
        missing_points=_as_list(data.get("missing_points")),
        mistakes=_as_list(data.get("mistakes")),
        suggestion=data.get("suggestion"),
    )

In [33]:
def format_follow_up_angles(angles: list[str]) -> str:
    if not angles:
        return "无"

    return "\n".join(
        f"{i}. {angle}"
        for i, angle in enumerate(angles, start=1)
    )


def generate_followup_question(
    turn: InterviewTurn,
    materials: FollowupMaterials,
    llm: ChatOpenAI | None = None,
) -> str:
    if not turn.main_answer or not turn.main_answer.strip():
        raise ValueError("主回答不能为空，不能生成追问")

    llm = llm or build_llm()

    prompt = ChatPromptTemplate.from_template(InterviewPromptSet.followup_prompt)
    chain = prompt | llm

    response = chain.invoke(
        {
            "main_question": turn.main_question,
            "main_answer": turn.main_answer,
            "expected_answer": materials.expected_answer or "无",
            "follow_up_angles": format_follow_up_angles(materials.follow_up_angles),
            "retrieved_context": materials.retrieved_context or "无",
        }
    )

    content = response.content.strip()

    if not content:
        raise ValueError("LLM 没有生成追问")

    return content

In [34]:
session = InterviewEngine.submit_main_answer(session, "我会使用 std::async 创建异步任务，并通过 std::future 获取结果。实际项目中主要关注任务粒度和 get 阻塞问题。",llm)

In [35]:
print(session.session_id)
print(session.plan)
print(session.plan.items[0])
print(session.updated_time)
print(session.status)

turn = session.turns[0]
print(turn.main_question)
print(turn.main_answer)
print(turn.followup_question)
print(turn.status)

c6a0e59b-c317-45f6-9364-cd6c77782cba
InterviewPlan(plan_id='f3e24206-d05f-44a2-8786-1a875646ad02', topics=['C++11', 'C++14'], num_questions=5, items=[InterviewPlanItem(index=0, question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', question=Question(question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', difficulty='medium', question='请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。', expected_answer='选择`std::async`而非手动创建线程的原因，如资源管理与异常安全；使用`std::future`获取结果时遇到的阻塞问题及如何通过`std::shared_future`优化；性能瓶颈分析，如线程池资源不足或任务粒度不当；优化措施，如调整任务队列或结合C++11的移动语义减少拷贝；复盘：权衡异步与同步方案的适用场景', follow_up_angles=['如何处理`std::future`的异常传播？', '在高并发场景下，如何设计任务调度器以避免线程饥饿？', 'C++11的异步机制与后续版本（如C++17的并行算法）有何改进？'])), InterviewPlanItem(index=1, question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', question=Question(question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', difficulty='medium', question='请描述一个你使用 C++14 特性（如泛型 lambda、变量模板或 conste

In [177]:
followup = InterviewEngine.get_current_question(session)
print(followup)

原问题:请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。
 原问题回答:异常会在 future.get() 的时候重新抛出，所以调用方需要捕获


In [183]:
session = InterviewEngine.submit_followup_answer(session, "我的追问回答...")

In [178]:
print(session.session_id)
print(session.plan)
print(session.plan.items[0])
print(session.updated_time)
print(session.status)

turn = session.turns[0]
print(turn.main_question)
print(turn.main_answer)
print(turn.followup_question)
print(turn.status)

turn = session.turns[1]
print(turn.main_question)
print(turn.status)

turn = session.turns[2]
print(turn.main_question)
print(turn.status)

78b1a06c-aba4-4211-9a01-3b2a3b149743
InterviewPlan(plan_id='49d1e757-d522-49c6-bff8-dec0f30ea3c7', topics=['C++11', 'C++14'], num_questions=5, items=[InterviewPlanItem(index=0, question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', question=Question(question_id='cpp_c++11_project_deep_dive_310fd31a76', topic='C++11', difficulty='medium', question='请描述你在项目中使用C++11的`std::future`和`std::async`实现异步任务调度的具体经历。请重点说明方案选择的原因、遇到的性能瓶颈及优化过程。', expected_answer='选择`std::async`而非手动创建线程的原因，如资源管理与异常安全；使用`std::future`获取结果时遇到的阻塞问题及如何通过`std::shared_future`优化；性能瓶颈分析，如线程池资源不足或任务粒度不当；优化措施，如调整任务队列或结合C++11的移动语义减少拷贝；复盘：权衡异步与同步方案的适用场景')), InterviewPlanItem(index=1, question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', question=Question(question_id='cpp_c++14_project_deep_dive_4d9f6311e8', topic='C++14', difficulty='medium', question='请描述一个你使用 C++14 特性（如泛型 lambda、变量模板或 constexpr 改进）优化过的真实项目模块。重点说明你为何选择该特性、如何权衡设计决策、遇到的性能或兼容性问题，以及最终的复盘结果。', expected_answer='明确项目背景：例如一个高性能计算库或网络服务模块，使用 C+

In [179]:
materials = build_materials_from_turn(session, turn)
llm = build_llm()
print(materials)

valMaterials(expected_answer='解释右值引用（`T&&`）和左值引用的区别，以及 `std::move` 如何将左值转换为右值引用。；描述移动构造函数和移动赋值运算符的实现，如何避免不必要的资源拷贝。；讨论移动语义在资源管理类（如 `std::unique_ptr`）中的应用，以及如何确保异常安全。；对比 C++11 移动语义与 C++98/03 的拷贝语义在性能上的差异。', reference_points=[], common_mistakes=[], rubric={}, retrieved_chunks=[])


In [ ]:
result = evaluate_turn(turn, materials, llm)

print(result)

valResult(score=10, reason='候选人完全没有回答题目要求，仅提到future.get()会抛出异常这一常识性细节，未描述任何项目中使用std::future和std::async的具体经历、方案选择原因、性能瓶颈及优化过程，答非所问。', hit_points=[], missing_points=['未讨论异步任务调度的具体实现场景', '未解释选择std::future和std::async的原因', '未描述性能瓶颈及优化过程', '未涉及与线程池或其他异步模型的比较', '未讨论异常处理、资源管理或性能调优'], mistakes=['回答内容与题目要求的核心内容完全无关'], suggestion='请围绕题目要求，详细描述你在项目中使用std::future和std::async的具体经验，包括为什么选择它们、遇到了什么性能问题、如何分析和优化的。同时补充实际代码示例或架构设计思路。')


In [16]:
topic = "C++11"
difficulty = "medium"
where = {
        "$and":[
            {"topic":topic},
            {"difficulty":difficulty}
        ]
}
result = db.get(where = where,
                include = ["metadatas"])
print(type(result))
print(result)
print(result.get("ids",0))
print(type(result.get("ids",0)))
ids = result.get("ids",0)
print(ids)
print(type(ids))

all_candidata_ids = []
all_candidata_ids.extend(ids)
print(all_candidata_ids)

<class 'dict'>
{'ids': ['all_questions_5', 'all_questions_6', 'all_questions_285', 'all_questions_289', 'all_questions_290', 'all_questions_293'], 'embeddings': None, 'documents': None, 'uris': None, 'included': ['metadatas'], 'data': None, 'metadatas': [{'role': 'cpp', 'source_file': 'cpp_project.jsonl', 'source': 'all_questions.jsonl', 'topic': 'C++11', 'question_type': 'project_deep_dive', 'doc_type': 'interview_question', 'topic_id': 'cpp:cxx11', 'question_id': 'cpp_c++11_project_deep_dive_310fd31a76', 'difficulty': 'medium', 'source_line': 5}, {'source_file': 'cpp_project.jsonl', 'question_type': 'project_deep_dive', 'role': 'cpp', 'doc_type': 'interview_question', 'difficulty': 'medium', 'question_id': 'cpp_c++11_project_deep_dive_556dc51861', 'source': 'all_questions.jsonl', 'source_line': 6, 'topic': 'C++11', 'topic_id': 'cpp:cxx11'}, {'source_line': 21, 'topic': 'C++11', 'role': 'cpp', 'difficulty': 'medium', 'source_file': 'cpp_technical.jsonl', 'topic_id': 'cpp:cxx11', 'doc_

In [17]:
topics = ["C++11","C++14"]
difficulty = "medium"
all_candidata_ids = []
topic_to_ids: dict[str, list[str]] = {}
for topic in topics:
    where = {
            "$and":[
                {"topic":topic},
                {"difficulty":difficulty}
            ]
    }
    result = db.get(where = where,
                    include = ["metadatas"])
    print(type(result))
    print(result)
    print(result.get("ids",0))
    print(type(result.get("ids",0)))
    ids = result.get("ids",0)
    print(ids)
    print(type(ids))

    all_candidata_ids.extend(ids)
    topic_to_ids[topic] = ids
    print(all_candidata_ids)
    print(topic_to_ids)

<class 'dict'>
{'ids': ['all_questions_5', 'all_questions_6', 'all_questions_285', 'all_questions_289', 'all_questions_290', 'all_questions_293'], 'embeddings': None, 'documents': None, 'uris': None, 'included': ['metadatas'], 'data': None, 'metadatas': [{'question_type': 'project_deep_dive', 'doc_type': 'interview_question', 'source_file': 'cpp_project.jsonl', 'topic_id': 'cpp:cxx11', 'source': 'all_questions.jsonl', 'role': 'cpp', 'source_line': 5, 'question_id': 'cpp_c++11_project_deep_dive_310fd31a76', 'topic': 'C++11', 'difficulty': 'medium'}, {'question_id': 'cpp_c++11_project_deep_dive_556dc51861', 'difficulty': 'medium', 'doc_type': 'interview_question', 'role': 'cpp', 'topic_id': 'cpp:cxx11', 'question_type': 'project_deep_dive', 'topic': 'C++11', 'source': 'all_questions.jsonl', 'source_file': 'cpp_project.jsonl', 'source_line': 6}, {'doc_type': 'interview_question', 'source_file': 'cpp_technical.jsonl', 'source_line': 21, 'question_id': 'cpp_c++11_technical_dd6a2d503d', 'que

In [18]:
import random
all_candidate_ids = list(dict.fromkeys(all_candidata_ids))
print(all_candidate_ids)
random.shuffle(all_candidate_ids)
print(all_candidate_ids)

['all_questions_5', 'all_questions_6', 'all_questions_285', 'all_questions_289', 'all_questions_290', 'all_questions_293', 'all_questions_7', 'all_questions_295', 'all_questions_296', 'all_questions_300', 'all_questions_302']
['all_questions_7', 'all_questions_290', 'all_questions_293', 'all_questions_300', 'all_questions_285', 'all_questions_5', 'all_questions_6', 'all_questions_295', 'all_questions_302', 'all_questions_289', 'all_questions_296']


In [41]:
selected_ids = ['all_questions_296', 'all_questions_293']
selected_result = db.get(
    ids = selected_ids,
    include = ["documents", "metadatas"]
)
print(selected_result)
print(type(selected_result))
print(selected_result.get("ids"))
print(selected_result.get("documents"))
print(selected_result.get("metadatas"))
print(type(selected_result.get("ids")))
print(type(selected_result.get("documents")))
print(type(selected_result.get("metadatas")))
print(selected_result.items())

{'ids': ['all_questions_293', 'all_questions_296'], 'embeddings': None, 'documents': ['[文档类型]: interview_question\n[题目ID]: cpp_c++11_technical_975279924f\n[岗位]: cpp\n[知识点]: C++11\n[主题ID]: cpp:cxx11\n[题型]: technical\n[难度]: medium\n[题目]: C++11 引入了 `std::shared_ptr` 和 `std::weak_ptr`，请解释它们的内存模型和循环引用问题，并说明如何在工程中结合 `std::enable_shared_from_this` 安全地管理对象生命周期。\n[参考答案]: 描述 `std::shared_ptr` 的引用计数机制和控制块结构。；解释循环引用如何导致内存泄漏，以及 `std::weak_ptr` 如何打破循环。；讨论 `std::enable_shared_from_this` 的使用场景和实现原理，确保对象在已有 `shared_ptr` 管理下安全创建新引用。；对比 `std::shared_ptr` 与原始指针在资源管理上的优缺点，及在多线程环境下的注意事项。\n', '[文档类型]: interview_question\n[题目ID]: cpp_c++14_technical_01c8d38518\n[岗位]: cpp\n[知识点]: C++14\n[主题ID]: cpp:cxx14\n[题型]: technical\n[难度]: medium\n[题目]: C++14引入了泛型lambda，请解释其工作原理，并给出一个在工程中使用泛型lambda处理异构容器的示例。\n[参考答案]: 泛型lambda允许lambda表达式使用auto参数，编译器自动推导类型，类似于模板函数。；工作原理：编译器将泛型lambda实例化为模板函数，支持延迟类型推导。；示例：使用std::for_each处理std::tuple，泛型lambda可接受任意类型参数。；工程优势：简化代码，避免显式模板定义，提高可读性和维护性。\n'], 'uris': None, 'included': ['documents', 